In [2]:
import cv2
import numpy as np
from ultralytics import YOLO

In [3]:
model = YOLO("yolov8n.pt")  

In [4]:
# Slot polygon (4 corners)
slot_polygon = np.array([
    [200, 300],
    [400, 300],
    [400, 500],
    [200, 500]
])


In [5]:
slot_center_line = [(200, 400), (400, 400)]
slot_angle = 0   # slot orientation in degrees (horizontal → 0°)

In [ ]:
cars = []
for box in r.boxes:
    if int(box.cls[0]) == 2:   # YOLO class 2 = car
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cars.append([x1, y1, x2, y2])

In [6]:
def get_center(x1, y1, x2, y2):
    return int((x1+x2)/2), int((y1+y2)/2)

In [7]:
def get_angle(box):
    pts = np.array(box, dtype=np.float32)

    # If fewer than 4 points → return angle = 0
    if pts.shape[0] < 4:
        return 0

    rect = cv2.minAreaRect(pts)
    return rect[2]


In [8]:
def point_to_line_distance(point, line_start, line_end):
    px, py = point
    x1, y1 = line_start
    x2, y2 = line_end

    num = abs((y2 - y1)*px - (x2 - x1)*py + x2*y1 - y2*x1)
    den = np.sqrt((y2 - y1)**2 + (x2 - x1)**2)
    return num / den

In [10]:
cap = cv2.VideoCapture("video1.mp4") 

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    for r in results:
        for box in r.boxes:
            
            cls = int(box.cls[0])
            if cls != 2:   # YOLO class 2 = car
                continue

            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx, cy = get_center(x1, y1, x2, y2)

            # Car corner points for angle estimation
            car_box_pts = np.array([[x1, y1],[x2, y1],[x2, y2],[x1, y2]], dtype=np.float32)

            car_angle = get_angle(car_box_pts)

            # -----------------------------
            #   Geometric Checks
            # -----------------------------

            # 1. Center deviation from slot line
            center_dev = point_to_line_distance(
                (cx, cy), 
                slot_center_line[0], 
                slot_center_line[1]
            )

            # 2. Angle difference
            angle_diff = abs(car_angle - slot_angle)


            # 3. Boundary check
            inside = cv2.pointPolygonTest(slot_polygon, (cx, cy), False)

            # Decision logic
            aligned = (
                inside >= 0 and
                center_dev < 25 and
                angle_diff < 12
            )

            status_text = "ALIGNED" if aligned else "MISALIGNED"
            color = (0,255,0) if aligned else (0,0,255)

            # Draw display
            cv2.polylines(frame, [slot_polygon], True, (255,255,0), 2)
            cv2.circle(frame, (cx,cy), 5, color, -1)
            cv2.putText(frame, status_text, (int(x1), int(y1)-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    cv2.imshow("SmartPark - Alignment Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break



0: 384x640 3 cars, 1 cell phone, 116.9ms
Speed: 42.7ms preprocess, 116.9ms inference, 4.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 cell phone, 81.4ms
Speed: 4.0ms preprocess, 81.4ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 cell phone, 79.7ms
Speed: 3.4ms preprocess, 79.7ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 cell phone, 79.3ms
Speed: 3.0ms preprocess, 79.3ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 cell phone, 77.7ms
Speed: 2.2ms preprocess, 77.7ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 cell phone, 81.5ms
Speed: 3.1ms preprocess, 81.5ms inference, 2.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 cell phone, 80.7ms
Speed: 3.3ms preprocess, 80.7ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 1 cell phon

: 

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import requests

# ----------------------------
# Load YOLO
# ----------------------------
model = YOLO("yolov8n.pt")

# ----------------------------
# Parking Slot Geometry
# (Edit these 4 points for your video)
# ----------------------------
slot_polygon = np.array([
    [200, 300],
    [400, 300],
    [400, 500],
    [200, 500]
], dtype=np.int32)

slot_center_line = [(200, 400), (400, 400)]
slot_angle = 0  # horizontal slot

# ----------------------------
# Helper Functions
# ----------------------------

def get_center(x1, y1, x2, y2):
    return int((x1 + x2) / 2), int((y1 + y2) / 2)

def get_angle(box_pts):
    pts = np.array(box_pts, dtype=np.float32)
    if pts.shape[0] < 4:
        return 0
    rect = cv2.minAreaRect(pts)
    return rect[2]

def point_to_line_distance(point, line_start, line_end):
    px, py = point
    x1, y1 = line_start
    x2, y2 = line_end

    num = abs((y2 - y1) * px - (x2 - x1) * py + x2 * y1 - y2 * x1)
    den = np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)
    return num / den

# ----------------------------
# CCTV Feed
# ----------------------------
cap = cv2.VideoCapture("photo1.jpg")  # change to RTSP if needed

# ----------------------------
# Real-Time Loop
# ----------------------------
while True:

    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    # If no detections, skip
    if results[0].boxes is None or len(results[0].boxes) == 0:
        cv2.imshow("SmartPark", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    # ----------------------------
    # STEP 1: Collect all detected cars
    # ----------------------------
    cars = []
    for box in results[0].boxes:
        if int(box.cls[0]) == 2:  # class 2 = car
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cars.append([x1, y1, x2, y2])

    # If no cars found, continue
    if len(cars) == 0:
        cv2.imshow("SmartPark", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    # ----------------------------
    # STEP 2: Sort cars from LEFT to RIGHT
    # ----------------------------
    cars = sorted(cars, key=lambda c: c[0])  # sort by x1 (left side)

    # ----------------------------
    # STEP 3: Select only the first 2 cars
    # ----------------------------
    cars_to_check = cars[:2]

    # ----------------------------
    # STEP 4: Run alignment check on these two cars
    # ----------------------------
    for (x1, y1, x2, y2) in cars_to_check:

        cx, cy = get_center(x1, y1, x2, y2)

        car_box_pts = np.array([
            [x1, y1],
            [x2, y1],
            [x2, y2],
            [x1, y2]
        ], dtype=np.float32)

        car_angle = get_angle(car_box_pts)

        center_dev = point_to_line_distance(
            (cx, cy),
            slot_center_line[0],
            slot_center_line[1]
        )

        angle_diff = abs(car_angle - slot_angle)

        inside = cv2.pointPolygonTest(slot_polygon, (cx, cy), False)

        aligned = (
            inside >= 0 and
            center_dev < 25 and
            angle_diff < 12
        )

        status_text = "MISALIGNED" if aligned else "ALIGNED"
        color = (0, 255, 0) if aligned else (0, 0, 255)

        # Draw detections
        cv2.polylines(frame, [slot_polygon], True, (255, 255, 0), 2)
        cv2.circle(frame, (cx, cy), 5, color, -1)

        cv2.putText(frame, status_text, (int(x1), int(y1) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        # Send to backend (optional)
        payload = {
            "slot_id": 1,
            "alignment": "aligned" if aligned else "misaligned"
        }
        # requests.post("http://localhost:8000/update-status", json=payload)

    # Show output
    cv2.imshow("SmartPark", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()



0: 384x640 2 cars, 102.7ms
Speed: 6.8ms preprocess, 102.7ms inference, 5.6ms postprocess per image at shape (1, 3, 384, 640)


: 

In [4]:
import cv2
import numpy as np
import math
from ultralytics import YOLO

# ----------------------------
# CONFIG (tune these)
# ----------------------------
MODEL_PATH = "yolov8n.pt"       # yolov8 model file
VIDEO_PATH = "video.mp4"       # your video or RTSP stream

# Three slot polygons (example tuned for the reference image).
# Edit these pixel coordinates for your camera/frame if needed.
SLOTS = {
    1: np.array([[60, 140], [240, 140], [240, 420], [60, 420]], dtype=np.int32),
    2: np.array([[260, 140], [440, 140], [440, 420], [260, 420]], dtype=np.int32),
    3: np.array([[460, 140], [640, 140], [640, 420], [460, 420]], dtype=np.int32),
}

# Slot center-lines (choose two points from each slot polygon or compute centroid-line)
# We'll compute slot centroid and slot orientation from polygon automatically.
CONFIDENCE_THRESHOLD = 0.5     # >= aligned, < misaligned
ANGLE_MAX_DEG = 30.0           # angle considered worst-case (normalized)
CENTER_MAX_RATIO = 0.5         # relative to slot half-width (worst-case)
WEIGHT_CENTER = 0.45
WEIGHT_ANGLE = 0.45
WEIGHT_INSIDE = 0.10
SHOW_DEBUG = True

# ----------------------------
# Helper functions
# ----------------------------
def load_model(path):
    return YOLO(path)

def get_center(x1, y1, x2, y2):
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

def safe_min_area_angle(box_pts):
    pts = np.array(box_pts, dtype=np.float32)
    if pts.shape[0] < 3:
        return 0.0
    rect = cv2.minAreaRect(pts)   # ((cx,cy),(w,h),angle)
    angle = rect[2]
    w, h = rect[1]
    # convert to canonical 0..180
    if w < h:
        angle = angle + 90
    angle = angle % 180
    return angle

def polygon_centroid(polygon):
    pts = polygon.reshape((-1,2))
    x = pts[:,0].mean()
    y = pts[:,1].mean()
    return (x, y)

def polygon_orientation_deg(polygon):
    # approximate orientation as angle of longest side using convex hull
    pts = polygon.reshape((-1,2)).astype(np.float32)
    hull = cv2.convexHull(pts)
    if hull is None or hull.shape[0] < 2:
        return 0.0
    # find longest edge in hull
    hull_pts = hull.reshape((-1,2))
    max_len = 0
    best_angle = 0.0
    n = len(hull_pts)
    for i in range(n):
        p1 = hull_pts[i]
        p2 = hull_pts[(i+1) % n]
        dx = p2[0] - p1[0]
        dy = p2[1] - p1[1]
        length = math.hypot(dx, dy)
        if length > max_len:
            max_len = length
            best_angle = math.degrees(math.atan2(dy, dx)) % 180
    return best_angle

def minimal_angle_diff_deg(a, b):
    diff = abs(a - b) % 180
    if diff > 90:
        diff = 180 - diff
    return diff

def point_to_line_distance(point, line_start, line_end):
    px, py = point
    x1, y1 = line_start
    x2, y2 = line_end
    num = abs((y2 - y1) * px - (x2 - x1) * py + x2 * y1 - y2 * x1)
    den = math.hypot((y2 - y1), (x2 - x1))
    return (num / den) if den != 0 else float('inf')

def normalize(val, max_val):
    if max_val <= 0:
        return 1.0 if val <= 0 else 0.0
    v = val / max_val
    v = max(0.0, min(1.0, v))
    return v

# ----------------------------
# Precompute slot metadata
# ----------------------------
slot_meta = {}
for sid, poly in SLOTS.items():
    centroid = polygon_centroid(poly)
    orientation = polygon_orientation_deg(poly)
    xs = poly[:,0]
    half_width = (xs.max() - xs.min()) / 2.0 if xs.max() - xs.min() > 0 else 1.0
    slot_meta[sid] = {
        "poly": poly,
        "centroid": centroid,
        "orientation": orientation,
        "half_width": half_width
    }

# load model
model = load_model(MODEL_PATH)

# ----------------------------
# Video capture
# ----------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

# main loop
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)
    # handle no detection
    if results[0].boxes is None or len(results[0].boxes) == 0:
        # draw slots and show
        for sid, meta in slot_meta.items():
            #cv2.polylines(frame, [meta["poly"]], True, (255,255,0), 2)
            cx, cy = int(meta["centroid"][0]), int(meta["centroid"][1])
            cv2.putText(frame, f"Slot {sid}: NO CAR", (cx-40, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 2)
        cv2.imshow("SmartPark - Multi Slot", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    # collect detected cars (class id 2 for COCO car)
    cars = []
    for box in results[0].boxes:
        try:
            cls = int(box.cls[0])
        except:
            continue
        if cls == 2:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cars.append({"bbox": [x1,y1,x2,y2]})

    # if no cars, draw slots and continue
    if len(cars) == 0:
        for sid, meta in slot_meta.items():
            #cv2.polylines(frame, [meta["poly"]], True, (255,255,0), 2)
            cx, cy = int(meta["centroid"][0]), int(meta["centroid"][1])
            cv2.putText(frame, f"Slot {sid}: NO CAR", (cx-40, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 2)
        cv2.imshow("SmartPark - Multi Slot", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        continue

    # compute car centers
    for c in cars:
        x1,y1,x2,y2 = c["bbox"]
        cx, cy = get_center(x1,y1,x2,y2)
        c["center"] = (cx, cy)
        # keep bbox points for angle estimation
        c["box_pts"] = np.array([[x1,y1],[x2,y1],[x2,y2],[x1,y2]], dtype=np.float32)

    # assign each car to the nearest slot centroid (one car can be nearest to same slot — that's fine for now)
    for c in cars:
        cx, cy = c["center"]
        best_sid = None
        best_dist = float('inf')
        for sid, meta in slot_meta.items():
            sx, sy = meta["centroid"]
            d = math.hypot(cx - sx, cy - sy)
            if d < best_dist:
                best_dist = d
                best_sid = sid
        c["assigned_slot"] = best_sid

    # create a mapping slot -> list of cars assigned
    slot_assignments = {sid: [] for sid in slot_meta.keys()}
    for c in cars:
        slot_assignments[c["assigned_slot"]].append(c)

    # evaluate each slot (if no car assigned -> show NO CAR)
    for sid, meta in slot_meta.items():
        poly = meta["poly"]
        centroid = meta["centroid"]
        slot_orientation = meta["orientation"]
        slot_half_width = meta["half_width"]

        assigned = slot_assignments.get(sid, [])

        # draw slot polygon
        cv2.polylines(frame, [poly], True, (255,255,0), 2)
        cx_s, cy_s = int(centroid[0]), int(centroid[1])

        if len(assigned) == 0:
            cv2.putText(frame, f"Slot {sid}: NO CAR", (cx_s-40, cy_s), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 2)
            continue

        # if multiple cars assigned (rare), choose the one with smallest distance to centroid
        if len(assigned) > 1:
            assigned = sorted(assigned, key=lambda c: math.hypot(c["center"][0]-centroid[0], c["center"][1]-centroid[1]))
        car = assigned[0]
        x1,y1,x2,y2 = car["bbox"]
        car_cx, car_cy = car["center"]

        # compute car angle
        car_angle = safe_min_area_angle(car["box_pts"])
        angle_diff = minimal_angle_diff_deg(car_angle, slot_orientation)

        # center deviation: distance from slot centroid projected onto slot center-line
        # we'll use distance to slot centroid as proxy, normalized by half-width
        center_dev_px = math.hypot(car_cx - centroid[0], car_cy - centroid[1])
        center_dev_norm = normalize(center_dev_px, slot_half_width * CENTER_MAX_RATIO)

        # angle normalized
        angle_norm = normalize(angle_diff, ANGLE_MAX_DEG)

        # inside polygon test
        inside = cv2.pointPolygonTest(poly, (car_cx, car_cy), False)
        inside_score = 1.0 if inside >= 0 else 0.0

        # convert errors -> confidences (1=perfect, 0=worst)
        center_conf = max(0.0, 1.0 - center_dev_norm)
        angle_conf = max(0.0, 1.0 - angle_norm)

        confidence = WEIGHT_CENTER * center_conf + WEIGHT_ANGLE * angle_conf + WEIGHT_INSIDE * inside_score

        aligned = confidence >= CONFIDENCE_THRESHOLD

        color = (0,255,0) if aligned else (0,0,255)
        label = f"Slot {sid}: {'ALIGNED' if aligned else 'MISALIGNED'} ({confidence:.2f})"

        # draw car bbox and info
        cv2.rectangle(frame, (int(x1),int(y1)), (int(x2),int(y2)), color, 2)
        cv2.circle(frame, (int(car_cx), int(car_cy)), 4, color, -1)
        cv2.putText(frame, label, (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        if SHOW_DEBUG:
            dbg1 = f"angCar={car_angle:.1f} angSlot={slot_orientation:.1f} dAng={angle_diff:.1f}"
            dbg2 = f"ctrDevPx={center_dev_px:.1f} norm={center_dev_norm:.2f} inside={int(inside)}"
            cv2.putText(frame, dbg1, (int(x1), int(y2)+18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
            cv2.putText(frame, dbg2, (int(x1), int(y2)+36), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)

    # show frame
    cv2.imshow("SmartPark - Multi Slot", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 3 cars, 66.4ms
Speed: 3.0ms preprocess, 66.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 64.4ms
Speed: 2.2ms preprocess, 64.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 67.3ms
Speed: 2.6ms preprocess, 67.3ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 67.3ms
Speed: 2.7ms preprocess, 67.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 62.6ms
Speed: 1.8ms preprocess, 62.6ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 57.0ms
Speed: 2.1ms preprocess, 57.0ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 53.9ms
Speed: 2.0ms preprocess, 53.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 cars, 50.9ms
Speed: 2.3ms preprocess, 50.9ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x